In [52]:
from dotenv import load_dotenv
load_dotenv() 

True

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent


In [54]:
loader = PyPDFLoader("../data/Artificial Intelligence Overview.pdf")
docs = loader.load()
len(docs)

2

In [55]:
splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
split_docs = splitter.split_documents(docs)
len(split_docs)

2

In [56]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [ ]:
vector_store = InMemoryVectorStore.from_documents(
    documents=split_docs, 
    embedding=embeddings, 
    collection_name="ai_docs"
    )

In [58]:
same_record = vector_store.similarity_search("Gen AI")
len(same_record)

2

In [59]:
same_record

[Document(id='0a63f23b-183f-4f4b-b8fe-fb7d68f2bf4d', metadata={'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-10-01T08:34:45+00:00', 'title': 'Artificial Intelligence: Executive Overview', 'moddate': '2026-10-01T08:34:45+00:00', 'source': '../data/Artificial Intelligence Overview.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Artificial Intelligence\nA concise brieﬁng on architectural paradigms, industry deployment metrics, and capabilities.\nEXECUTIVE SUMMARY\nArtiﬁcial Intelligence (AI) represents the simulation of human intelligence by computer systems. Modern\ndevelopments are driven by deep learning networks, large language models (LLMs), and scalable compute\ninfrastructure, enabling rapid transition from research frameworks to critical enterprise operations.\nMarket & Growth Snapshot\n37.3%\nCAGR (2023–2030)\nGlobal Market Expansion\n72%\nENTERPRISE ADOPTION\nIntegrating AI in ≥1 Function\n10x\nCOMPUTE SCALING\nModel Training Powe

> we need to create ai agent. who fetch similar data from vector db based on query and also give result

#### agent -> call tools -> llm -> prompt

In [60]:
@tool
def retiever_tool(query:str):
    """
    This tool can help you retrieve the relevant data of the 
    PDF document.
    """
    docs = vector_store.similarity_search(query=query, k=5)
    print(len(docs), docs[0].page_content)

    context = ""
    for doc in docs:
        context = doc.page_content + '\n\n'

    return context    



In [61]:
# retiever_tool.invoke("Data science")

In [62]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [63]:
prompt = """
    You are a helpful AI assistant.
    Use the 'retiever_tool' tool to answer the user questions. 
    If you don't know the answer,
    just say that you don't know, 
    don't try to make up an answer.
    """

In [64]:
agent = create_agent(
    model = llm,
    tools=[retiever_tool],
    system_prompt=prompt
)

In [65]:
query = "Artificial Intelligence content"

In [66]:
resposne = agent.invoke(
    {"messages":[{"role":"user", "content": query}]}
)

2 Artiﬁcial Intelligence Overview Page 1 of 1


In [67]:
resposne

{'messages': [HumanMessage(content='Artificial Intelligence content', additional_kwargs={}, response_metadata={}, id='47800a3d-2144-4630-8594-478d34a89838'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'retiever_tool', 'arguments': '{"query": "Artificial Intelligence content"}'}, '__gemini_function_call_thought_signatures__': {'call_230533': 'EmAKXgFpFH0T0p3dosdAuBzxLqDH26V/0/Ds5ur5Wfm0TGpKV/girZcTGFGOidNyBV8tWzt03LhT3XlJwAhaz4Iz5NSzW1OK6bphXlbcwC5neA13HB/BN2nkrxxhYfHHZ/I='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f774-31f4-76f1-b2e1-0328cc0a7705-0', tool_calls=[{'name': 'retiever_tool', 'args': {'query': 'Artificial Intelligence content'}, 'id': 'call_230533', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 122, 'output_tokens': 19, 'total_tokens': 141, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(cont

In [68]:
print(resposne["messages"][-1].content[0]["text"])

Based on the provided document, here is a concise briefing on **Artificial Intelligence (AI)**:

### **Executive Summary**
Artificial Intelligence represents the simulation of human intelligence by computer systems. Modern developments are driven by deep learning networks, large language models (LLMs), and scalable compute infrastructure, enabling a rapid transition from research frameworks to critical enterprise operations.

### **Market & Growth Snapshot**
* **37.3% CAGR (2023–2030):** Global market expansion.
* **72% Enterprise Adoption:** Integrating AI in $\ge 1$ function.
* **10x Compute Scaling:** Model training power increase.

---

### **Core Paradigms**
1. **Machine Learning (Predictive):** Statistical models that improve performance through experience without explicit programming. Essential for forecasting and structured data classification.
2. **Deep Learning (Perceptual):** Multi-layered neural networks inspired by human brain structure. Powers computer vision, speech synt